# 01 — Data Inventory (CICIDS2017)

**Goal:** inspect the raw data before any cleaning. We confirm the file count, true combined shape, exact label names, and the known CICIDS2017 defects (leading spaces in column names, infinite/NaN values).

**We do NOT clean or drop anything here — inspection only.**

After running, paste back: (1) file count, (2) combined shape, (3) full Label value counts, (4) NaN and infinite counts.

In [1]:
import glob, os
import numpy as np
import pandas as pd

# Point this at your extracted CICIDS2017 (MachineLearningCVE) folder.
DATA_DIR = r"D:\Capstone Project\MachineLearningCVE"

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"CSV files found: {len(files)}")
for f in files:
    print("  -", os.path.basename(f))

CSV files found: 8
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Wednesday-workingHours.pcap_ISCX.csv


In [2]:
# Load all files, stripping the well-known leading spaces in column names.
frames = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = df.columns.str.strip()          # fixes ' Label', ' Destination Port', etc.
    df["__source_file"] = os.path.basename(f)     # keep track of origin for now
    print(f"{os.path.basename(f):55s} shape={df.shape}")
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print("\nCOMBINED shape:", data.shape)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        shape=(225745, 80)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    shape=(286467, 80)
Friday-WorkingHours-Morning.pcap_ISCX.csv               shape=(191033, 80)
Monday-WorkingHours.pcap_ISCX.csv                       shape=(529918, 80)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv shape=(288602, 80)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  shape=(170366, 80)
Tuesday-WorkingHours.pcap_ISCX.csv                      shape=(445909, 80)
Wednesday-workingHours.pcap_ISCX.csv                    shape=(692703, 80)

COMBINED shape: (2830743, 80)


In [3]:
# Exact label names and counts.
print("Label value counts:")
print(data["Label"].value_counts())

Label value counts:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [4]:
# Data-quality check (the issues our preprocessing must handle).
numeric = data.select_dtypes(include=[np.number])
print("Total NaN cells:", int(data.isna().sum().sum()))
print("Total +/- infinite cells:", int(np.isinf(numeric.to_numpy()).sum()))

Total NaN cells: 1358
Total +/- infinite cells: 4376


---
### What to report back
1. Did it find **8** files and run without errors?
2. The **combined shape** (rows x columns).
3. The full **Label value counts**.
4. The **NaN** and **infinite** cell counts.

If you hit an error (encoding on the Thursday Web Attacks file, memory, etc.), paste it instead and we'll fix it before moving on.